In [ ]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

include("../src/flexOPT.jl")

using .commonBatchs, .planet1D, .GeoPoints, .flexOPT

# model construction

In [ ]:
using Symbolics,CairoMakie
numPointsX = collect(2:2)
logsOfHinverse = [1.0*i for i in 0:3]

cases=[]
#prefix="B"*string(tmpOrderBspace)*"_"*"w"*string(tmpWorderBspace)*"_"*string(tmpSupplementaryOrder)*"_"
prefix=""
L = 10.0*π
cases = push!(cases,(name=prefix*"λ_2",u=cos(x),β=cos(x).^2 + 1))

@variables x
∂ = Differential(x)


misfit = Array{Float64,3}(undef,length(logsOfHinverse),length(cases),length(numPointsX))

modelFamily=[]

fig = Figure()
ax = Axis(fig[1, 1], xlabel="x", ylabel="model")
Nx=1000
Δx=L/(Nx-1)
X = [Δx * (i-1) for i ∈ range(1,Nx)]

for iCase ∈ eachindex(cases)
    name,_,β = cases[iCase]
    models=[]
    model=[Symbolics.value(substitute(β,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    models=push!(models, model)

    for iH ∈ eachindex(logsOfHinverse)
        modelName = name*string(Nx)
        ΔxTry = exp(-logsOfHinverse[iH])
        Nx = Int(L÷ΔxTry) +1
        Δx = L/(Nx-1)
        modelPoints = (Nx)
        tmpModel = (models=models, modelName=modelName, modelPoints=modelPoints)
        modelFamily=push!(modelFamily,tmpModel)
    end

    lines!(ax, X, models[end])
end
display(fig)
for iPointsUsed ∈ eachindex(numPointsX), iCase ∈ eachindex(cases), iH ∈ eachindex(logsOfHinverse)
    name,u,β = cases[iCase]
    
    models=modelFamily[iCase]
    q = mySimplify(β*∂(u))
    qₓ = mySimplify(∂(q))
    
    iExperiment = (iH=iH,iCase=iCase,iPointsUsed=iPointsUsed)
    
    force = [Symbolics.value(substitute(qₓ,Dict(x=>X[i]))) for i ∈ range(1,Nx)]
    

    

    
end


# input parameters

In [ ]:

famousEquationType="1DpoissonHetero" #GT98
Δ = (1.0)

In [ ]:


# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=1
orderBspace=1

pointsInSpace=3
pointsInTime=3

# the order of errors to be controlled
supplementaryOrder=2

# new parameters for interpolated Taylor expansion μ for field

fieldItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1) #offsetSpace and offsetTime ∈ z 
# μ points should be distributed from y_min+offset*Δy to y_max-offset*Δy offset can be negative too


# new parameters for interpolated Taylor expansion μᶜ for material
materItpl = (ptsSpace = 1,ptsTime = 1,offsetSpace=1,offsetTime=1,YorderBspace=1,YorderBtime=1)



In [ ]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δ orderBtime orderBspace pointsInSpace pointsInTime supplementaryOrder fieldItpl materItpl


In [ ]:
optRec=myProduceOrLoad(makeOPTsemiSymbolic,concreteParametersForOPTConstruction,"semiSymbolic")

In [ ]:

function quasiNumericalOperatorConstruction(optRec,modelFam;absorbingBoundaries=nothing,maskedRegionInSpace=nothing)

    #region general introduction
    #
    # 01/05/2026 Nobuaki Fuji
    #
    # after the construction of local (semi-)symbolic expressions with linearised operators#
    # here we will read the model parameters and construct the numerical operators
    #
    # Nobuaki Fuji @ IPGP/UPC/IUF since 2024
    #
    # 
    # encouraged by Thibault Duretz @ U. Frankfurt Goethe, Kurama Okubo @ NIED
    #
    # Julia hackathon October 2024, March 2025
    #
    #

    # coordinates: Model: the real model domain; Whole: computation domain with absorbing boundaries; 
    #              Empty: Whole + some more points to avoid missing reference to the field and material (they should be just zeros)
    #

    # intermediate presentations: IPGP-CIA workshop October 2024; IPGP-ERI workshop November 2024; lighttalk @ systemI December 2024
    #             EGU @ Vienna May 2025
    #    
    #     Fuji & Duretz in preparation
    #
    #
    #
    #
    #endregion

    @unpack models, modelName, modelPoints = modelFam


    @unpack lhs, rhs, nodes, centresIndices, numbersOfTheSystem,fieldNames=optRec["recette"]
    Av=lhs.AjiννᶜU
    varM=lhs.varM
    Ulocal=lhs.Ulocal
    
    Γg=rhs.ΓjiννᶜF
    varF=rhs.varF
    Flocal=rhs.Flocal

    @unpack fields, extfields = fieldNames

    @unpack timeMarching,nConfigurations,nCoordinates,NtypeofExpr,NtypeofFields,NtypeofMaterialVariables=numbersOfTheSystem.numbersOfTheSystemL
    NtypeofExtFields=numbersOfTheSystem.numbersOfTheSystemL.NtypeofFields
    NtypeofExtMaterialVariables=numbersOfTheSystem.numbersOfTheSystemR.NtypeofMaterialVariables
    
    # normally the geometry configurations should be proposed in the preferred order
    @show nGeometry=nConfigurations
    Ndimension=nCoordinates

    

    # the last coordinate should be cosidered as time

    localPointsIndices = Vector{Any}(undef, nGeometry)
    middlepoints = Vector{Any}(undef, nGeometry)
    lhs_CartesianDependencies = lhs.CartesianDependencies
    rhs_CartesianDependencies = rhs.CartesianDependencies

    newD=Ndimension
    if !timeMarching
        newD = Ndimension + 1
        #tmpT = Symbolics.variable(timeDimensionString)
        modelPoints = (modelPoints..., 1)
        lhs_CartesianDependencies = vcat(lhs.CartesianDependencies,zeros(Int, 1, NtypeofMaterialVariables),)
        rhs_CartesianDependencies = vcat(rhs.CartesianDependencies,zeros(Int, 1, NtypeofMaterialVariables),)
        for iGeometry ∈ 1:nGeometry
            localPointsIndices[iGeometry] = [SVector{newD,Int}(p..., 1) for p in nodes[iGeometry]]
            middlepoints[iGeometry] = [SVector{newD,Int}(p..., 1) for p in nodes[iGeometry][centresIndices[iGeometry]]]
        end
    else
        for iGeometry ∈ 1:nGeometry
            localPointsIndices[iGeometry] = nodes[iGeometry]
            middlepoints[iGeometry] = nodes[iGeometry][centresIndices[iGeometry]]
        end
    end


    
    if length(models) !== NtypeofMaterialVariables 
        @error "Each material has to have its own model"
    end
    
    Models=Array{Any,1}(undef,NtypeofMaterialVariables)
    ModelPoints=Array{Int,2}(undef,Ndimension,NtypeofMaterialVariables)
    
    for iVar ∈ 1:NtypeofMaterialVariables
        CartesianDependency=lhs_CartesianDependencies[:,iVar]
        if ndims(models[iVar]) === CartesianDependency
            @error "model parameter dimension is not what you declared in the equation!"
        end
        if sum(CartesianDependency) == 0 # when it is a constant
            tmpModel=Array{Any,Ndimension}(undef,(ones(Int, Ndimension)...)...)
            ModelPoints[:,iVar] = ones(Int, Ndimension)
            tmpModel[vec2car(ones(Int, Ndimension))] = models[iVar]
            Models[iVar]=tmpModel
        else
            newCoords=expandVectors(size(models[iVar]),CartesianDependency)
            ModelPoints[:,iVar] = newCoords
            tmpModel=reshape(models[iVar],newCoords...)
            Models[iVar]=tmpModel

            for iCoord in eachindex(newCoords)
                if newCoords[iCoord]!== modelPoints[iCoord] && newCoords[iCoord] !== 1
                    @error "the model should have the same dimension! (or constant)"
                end
            end
        end
    end
    
    #endregion
    @show Models, ModelPoints


    
    #region construction of the fields

    wholeRegionPoints = nothing

    if absorbingBoundaries === nothing
        wholeRegionPoints=modelPoints
        absorbingBoundaries = zeros(Int,2, Ndimension)
    elseif absorbingBoundaries === "CerjanBoundary"
        wholeRegionPoints=modelPoints
        absorbingBoundaries = ones(Int,2, Ndimension-1)*CerjanGridPoints
        absorbingBoundaries=[absorbingBoundaries; 0 0]
        wholeRegionPoints=modelPoints.+sum(absorbingBoundaries,1) 
    else
        # absorbingBoundaries should be two column array 
        if size(absorbingBoundaries)[1] !== 2
            @error "you have to give us the left and right values for absorbing boundaries"
        elseif size(absorbingBoundaries)[2] !== size(modelPoints)[1] && !timeMarching
            @error "you have to give us the values for each direction for absorbing boundaries"
        elseif size(absorbingBoundaries)[2] === size(modelPoints)[1]-1 && timeMarching
            absorbingBoundaries=[absorbingBoundaries; 0 0]
        end
        wholeRegionPoints=modelPoints.+ sum(absorbingBoundaries, dims=1)[:]
    end
    wholeRegionPointsSpace=wholeRegionPoints[1:end-1]

    #endregion


    #region 

    # Preferred geometry at each spatial point.
    # Later this can be filled from a table or a classifier.
    geometryPreference = fill(1, Tuple(wholeRegionPointsSpace))

    spacePointsUsed=Vector{Any}(undef, nGeometry)
    timePointsUsedForOneStep=Vector{Any}(undef, nGeometry)
    for iGeometry ∈ 1:nGeometry
        localPointVecs = localPointsIndices[iGeometry]
        spacePointsUsed[iGeometry] = localPointVecs[end][1:end-1]
        timePointsUsedForOneStep[iGeometry] = localPointVecs[end][end]
    end

    # For now, all points use geometry 1, so this is the active time depth.
    activeTimePoints = timePointsUsedForOneStep[1]

    # Fields
 
    場 = Array{Any,2}(undef, NtypeofFields, activeTimePoints)

    for it ∈ 1:activeTimePoints
        for iField ∈ 1:NtypeofFields
            newstring = split(string(fields[iField]), "(")[1] * "_mod_t=" * string(it)
            場[iField, it] = Symbolics.variables(
                Symbol(newstring),
                Base.OneTo.(Tuple(wholeRegionPointsSpace))...
            )
        end
    end


    #since everything is super clumsy, here we make several useful functions to change one coordinate to another
    
    conv=spaceCoordinatesConversionfunctions(absorbingBoundaries[:,1:end-1], Ndimension-1)
    #endregion 

    # Useful point lists
    PointsSpace = CartesianIndices(Tuple(wholeRegionPointsSpace))
    NpointsSpace = length(PointsSpace)

    # For now, test functions are still identified with spatial points
    νWhole = collect(PointsSpace)

    # Pointwise preferred geometry and corresponding relative centre
    νGeometry = Vector{Int}(undef, NpointsSpace)
    νRelative = Vector{Any}(undef, NpointsSpace)

    for iPoint in eachindex(νWhole)
        νtmpWhole = νWhole[iPoint]
        iGeometry = geometryPreference[νtmpWhole]
        νGeometry[iPoint] = iGeometry

        # For now take the first centre of that geometry.
        # Later this can be refined if one point wants another centre within the same geometry.
        νRelative[iPoint] = middlepoints[iGeometry]
    end

    #endregion


    #region making a maskingField (for limited source areas, boundary conditions, etc.)

    maskingField=Array{Any,Ndimension-1}(undef,Tuple(wholeRegionPointsSpace)) # maskingField is defined only for whole domain
    champsLimité = nothing
    if maskedRegionInSpace === nothing
        maskingField .= 1.0
    elseif typeof(maskedRegionInSpace) === Array{CartesianIndex,1}
        champsLimité = Array{Any,2}(undef,NtypeofFields,timePointsUsedForOneStep)
        for it in 1:timePointsUsedForOneStep
            for iField in eachindex(fields)
                newstring=split(string(fields[iField]),"(")[1]*"_mod_limited"*"_t="*string(it)
                champsLimité[iField,it] = Array{Any,1}(undef,length(maskedRegionInSpace))
            end
        end
        maskingField .= 0.0
        tmpIndex=1
        for iSpace in maskedRegionInSpace
            jSpace = conv.model2whole(iSpace)
            maskingField[jSpace] =1.0
            for it in 1:timePointsUsedForOneStep
                for iField in eachindex(fields)
                    
                    #tmpChampsLimitéContents= (jSpace,場[iField,it][jSpace])
                    champsLimité[iField,it][tmpIndex]=場[iField,it][jSpace]
                end
            end
            tmpIndex += 1
        end
    else
        @error "maskedRegionInSpace should be a 1D array of CartesianIndex (if it is CartesianIndices, you need to collect(Tuple()))"
    end

    #endregion

    return (
        Models = Models,
        ModelPoints = ModelPoints,
        localPointsIndices = localPointsIndices,
        middlepoints = middlepoints,
        geometryPreference = geometryPreference,
        νGeometry = νGeometry,
        νRelative = νRelative,
        νWhole = νWhole,
        wholeRegionPoints = wholeRegionPoints,
        wholeRegionPointsSpace = wholeRegionPointsSpace,
        spacePointsUsed = spacePointsUsed,
        timePointsUsedForOneStep = timePointsUsedForOneStep,
        fields = fields,
        場 = 場,
    )

end


function spaceCoordinatesConversionfunctions(absorbingBoundaries, NdimensionMinusTime)
    offset_model = vec2car(absorbingBoundaries[1, 1:NdimensionMinusTime])
    #offset_empty = vec2car(spacePointsUsed)

    model2whole(a::CartesianIndex) = a + offset_model
    whole2model(a::CartesianIndex) = a - offset_model
    #whole2empty(a::CartesianIndex) = a + offset_empty
    #empty2whole(a::CartesianIndex) = a - offset_empty
    #model2empty(a::CartesianIndex) = whole2empty(model2whole(a))
    #empty2model(a::CartesianIndex) = whole2model(empty2whole(a))
    return(; model2whole, whole2model)
    #return (; model2whole, whole2model, whole2empty, empty2whole, model2empty, empty2model)
end

In [ ]:
quasiNumericalOperatorConstruction(optRec,modelFamily[1])